In this environment, the PPO policy does not output a discrete `buy/sell/hold` action. It outputs a continuous portfolio allocation vector:

[`env.py`](/Users/sumitghosh/application-sumit/Semanti-BTP/Sem-8/End-sem/one_notebook/src/env.py)

```python
self.action_space = spaces.Box(low=0.0, high=1.0, shape=(self.num_assets,), dtype=np.float32)
```

Then in [`env.py`](/Users/sumitghosh/application-sumit/Semanti-BTP/Sem-8/End-sem/one_notebook/src/env.py), `step()` converts that raw vector into normalized portfolio weights:

```python
action = np.clip(action, 1e-6, 1.0)
new_weights = action / np.sum(action)
```

So the true action is:

- one nonnegative weight per asset
- all weights sum to 1 after normalization
- no short selling
- no leverage
- fully invested at all times
- no explicit cash asset

So if there are `N` assets, PPO is choosing a point on the simplex:
`w = (w1, ..., wN)` with `wi >= 0` and `sum(wi)=1`.

**Without settlement delay**

If settlement were effectively immediate, the executed portfolio at time `t` would be the newly proposed `new_weights`. Conceptually:

- PPO observes the current state
- PPO outputs target weights
- those weights are applied immediately to today’s return and today’s turnover/cost

That means the agent has direct one-step control:
- action at time `t` changes exposure at time `t`
- action at time `t` also determines turnover versus old weights
- reward at time `t` reflects the consequence of that exact action

So the effective action space is exactly the normalized weight simplex.

**With `settlement_day = 2`**

The important logic is here in [`env.py`](/Users/sumitghosh/application-sumit/Semanti-BTP/Sem-8/End-sem/one_notebook/src/env.py):

```python
self.pending_actions.append(new_weights)
if len(self.pending_actions) == self.cfg.settlement_day:
    weights = self.pending_actions.popleft()
else:
    weights = self.weights
```

With `settlement_day = 2`:

- today’s chosen action is appended to a queue
- it is not executed immediately
- the portfolio uses an older queued action only when the queue length reaches 2

Operationally this means:

- at step `t`, PPO proposes `a_t`
- executed weights are still the old weights until the delay matures
- at step `t+1`, the action proposed earlier can become active

So the agent’s action space as an output space is unchanged, but the effective control space is delayed.

A useful way to think about it:

- PPO chooses `target weights`
- the environment executes `settled weights`
- with `settlement_day = 2`, `executed_weights_t` is approximately `action_{t-1}` after normalization, not `action_t`

**What this changes behaviorally**

Without delay:
- “I want 80% in asset A now” means exposure changes now.

With `settlement_day = 2`:
- “I want 80% in asset A now” means “apply this after one more step of waiting.”

So the policy is no longer just reacting to the present.
It must anticipate near-future returns, volatility, and sentiment.

**Turnover and cost are also delayed**

In the same `step()`:

```python
turnover = float(np.sum(np.abs(weights - self.weights)))
```

and later:

```python
self.weights = weights
```

So turnover is computed between:
- the executed settled weights for this step
- the previously active weights

That means transaction cost is charged when the delayed action actually becomes active, not when PPO first emits it.

**Why settlement_days does not directly enter the reward but still changes reward**

You’re exactly right that `settlement_day` does not appear explicitly in:

```python
reward = (
    ret_simple
    - self.lambda_risk * vol
    - self.drawdown_penalty * drawdown
    - self.turnover_penalty * turnover
    - self.cfg.transaction_cost * turnover
)
reward += 0.1 * sentiment_t * ret_simple
```

But it changes several inputs to this reward.

It indirectly changes reward through:

1. **Different executed weights**
   - `ret_log = dot(weights, returns[t])`
   - if `weights` are delayed, the portfolio return is based on stale allocation, not the latest action

2. **Different turnover timing**
   - turnover is computed when the delayed rebalance settles
   - this shifts when penalties and transaction costs hit

3. **Different drawdown path**
   - delayed exposure changes the portfolio value trajectory
   - that changes `max_value`, `drawdown`, and later rewards

4. **Different sentiment interaction**
   - the sentiment bonus multiplies `sentiment_t * ret_simple`
   - since `ret_simple` depends on delayed weights, the sentiment contribution changes too

So settlement delay changes the reward-generating state transition, even if the formula itself is unchanged.

**Simple intuition**

Suppose asset A will rise today and fall tomorrow.

Without settlement delay:
- PPO can shift into A today
- today’s reward benefits immediately

With `settlement_day = 2`:
- PPO’s action today may only affect tomorrow or later
- the agent can miss the rise and catch the fall instead
- same reward formula, different realized `ret_simple`

That is the core indirect effect.

**One subtle implementation point**

Because of this line:

```python
self.pending_actions = deque(maxlen=max(1, self.cfg.settlement_day))
```

and this logic:

```python
if len(self.pending_actions) == self.cfg.settlement_day:
    weights = self.pending_actions.popleft()
```

for `settlement_day = 2`, there is effectively a one-step execution lag after the queue fills. The action is not “lost,” but the environment is implementing delayed execution rather than instant rebalancing.

**Summary**

- The policy action space is always continuous target portfolio weights over assets.
- `settlement_day = 2` does not change the shape of the action space.
- It changes the mapping from chosen action to executed action.
- Therefore it changes returns, turnover, transaction cost timing, drawdown trajectory, and sentiment-adjusted reward indirectly.
- In PPO terms, it makes the environment more delayed and partially harder to control, because `a_t` affects future reward through delayed execution rather than immediate exposure.

If you want, I can also draw a step-by-step timeline for `settlement_day = 1` vs `2` vs `3` using a tiny 2-asset example.